In [7]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

warnings.filterwarnings("ignore")

COMMUTE_PATH = "D:/github/G23-Swindon-Borough-Council/commuting-data/data_swindon_with_lsoa_commute.csv"
FEATURES_PATH = "D:/github/G23-Swindon-Borough-Council/data preprocessing+EDA/features.csv"
TARGET = "log_total_GVA_2023"
RANDOM_STATE = 42

CAAFE_FEATURES = [
    "log_voa_rv_2023", "rv_per_working_age", "sme_density", "qualification_index",
    "firm_size_diversity", "rv_per_employee", "sme_qual_interaction",
    "employment_quality", "modern_sector_leverage", "asset_growth_diversity",
]
L3_QUALIFICATIONS = (
    "Level 3 qualifications: 2 or more A levels or VCEs, 4 or more AS levels, "
    "Higher School Certificate, Progression or Advanced Diploma, Welsh "
    "Baccalaureate Advance Diploma, NVQ level 3; Advanced GNVQ, City and Guilds "
    "Advanced Craft, ONC, OND, BTEC National, RSA Advanced Diploma %"
)
SCALE_SKILLS_BLOCK = [
    "total_employees", "part_time_employees", "total_enterprises_2025_msoa",
    "LU_large_2025_msoa", L3_QUALIFICATIONS,
]

In [8]:
commute = pd.read_csv(COMMUTE_PATH)
raw = pd.read_csv(FEATURES_PATH, low_memory=False).drop_duplicates("LSOA21CD")

# Use commuting SHARES only. The raw commute worker counts (workplace/inbound
# workers) duplicate the scale block and are non-actionable, so they are excluded;
# the shares carry the residential/workplace role signal and are the policy levers.
commute_features = [c for c in commute.columns if c.startswith("lsoa_") and "share" in c]
data = commute.merge(raw[["LSOA21CD"] + SCALE_SKILLS_BLOCK], on="LSOA21CD", how="left")
data = data.dropna(subset=[TARGET]).reset_index(drop=True)

extended = CAAFE_FEATURES + SCALE_SKILLS_BLOCK
feature_sets = {
    "Extended": extended,
    "Extended + commuting": extended + commute_features,
}

y = data[TARGET].to_numpy(dtype=float)
print("Swindon LSOAs", len(data), "| commuting share features", len(commute_features))

Swindon LSOAs 137 | commuting share features 7


In [9]:
SKEW_THRESHOLD = 1.5
WINSOR_LOWER, WINSOR_UPPER = 0.01, 0.99
COLLINEAR_THRESHOLD = 0.95


def fit_preprocessor(train_frame):
    log_columns = [
        c for c in train_frame.columns
        if train_frame[c].notna().any()
        and (train_frame[c].dropna() >= 0).all()
        and train_frame[c].skew() > SKEW_THRESHOLD
    ]
    logged = train_frame.copy()
    for c in log_columns:
        logged[c] = np.log1p(logged[c].clip(lower=0))
    medians = logged.median()
    logged = logged.fillna(medians)
    corr = logged.corr().abs()
    columns = list(logged.columns)
    drop = set()
    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            if columns[j] not in drop and corr.iloc[i, j] > COLLINEAR_THRESHOLD:
                drop.add(columns[j])
    kept = [c for c in columns if c not in drop]
    lower = logged[kept].quantile(WINSOR_LOWER)
    upper = logged[kept].quantile(WINSOR_UPPER)
    return {"log": log_columns, "medians": medians, "kept": kept,
            "lower": lower, "upper": upper}


def transform(frame, pp):
    out = frame.copy()
    for c in pp["log"]:
        out[c] = np.log1p(out[c].clip(lower=0))
    out = out.fillna(pp["medians"])[pp["kept"]]
    out = out.clip(pp["lower"], pp["upper"], axis=1)
    return out.to_numpy(dtype=float)


def prepare(train_frame, other_frame):
    pp = fit_preprocessor(train_frame)
    return transform(train_frame, pp), transform(other_frame, pp), len(pp["kept"])


def fit_predict(X_tr, y_tr, X_te):
    reg = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
    reg.fit(X_tr, y_tr)
    return np.asarray(reg.predict(X_te)).reshape(-1)


def score(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }

In [10]:
def cross_val(feature_names, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.empty(len(y))
    for tr, te in kf.split(data):
        X_tr, X_te, _ = prepare(data.loc[tr, feature_names], data.loc[te, feature_names])
        oof[te] = fit_predict(X_tr, y[tr], X_te)
    return oof


def holdout(feature_names, test_size=0.2):
    tr, te = train_test_split(np.arange(len(y)), test_size=test_size, random_state=RANDOM_STATE)
    X_tr, X_te, _ = prepare(data.loc[tr, feature_names], data.loc[te, feature_names])
    return y[te], fit_predict(X_tr, y[tr], X_te)


rows = []
for set_name, feature_names in feature_sets.items():
    oof = cross_val(feature_names)
    rows.append({"feature_set": set_name, "evaluation": "5-fold CV", **score(y, oof)})
    y_te, pred = holdout(feature_names)
    rows.append({"feature_set": set_name, "evaluation": "80/20 hold-out", **score(y_te, pred)})

results = pd.DataFrame(rows)

In [11]:
results.style.format({"R2": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}", "MAPE": "{:.2%}"})

,feature_set,evaluation,R2,MAE,RMSE,MAPE
0,Extended,5-fold CV,0.6889,0.4383,0.7190,11.83%
1,Extended,80/20 hold-out,0.7709,0.3919,0.5747,10.82%
2,Extended + commuting,5-fold CV,0.7012,0.4282,0.7047,11.61%
3,Extended + commuting,80/20 hold-out,0.7782,0.3759,0.5655,10.54%
